# 06 · Özgün model: KneeOrdinalNet tasarım günlüğü

**Önce GPU açın** (T4). ConvNeXt-Tiny omurgası + 4 veriye özgü bileşen. Sürümler sırayla eklenir:

| Sürüm | İçerik |
|---|---|
| Orig_v1 | çok ölçekli ortalama+maks havuzlama (B), CE+sampler |
| Orig_v2 | v1 + eklem hattı satır dikkati (C) |
| Orig_v3 | v2 + ordinal kayıp (D) |
| Orig_full | v3 + girişte kontrast standardizasyonu (A) |

- Hiperparametreler: ConvNeXt-Tiny'nin Optuna'daki en iyisi (adil kıyas, aynı omurga)
- Seed 42, 4 koşu, yaklaşık 35-40 dakika; biten koşular atlanır
- Yalnızca **validation** kullanılır, test yüklenmez

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 optuna==5.0.0
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## Mimari özeti (blok şeması için)

In [ ]:
from src import original as OR
from src.models import build_model, count_params
m = build_model('Original', pretrained=False)
arch = OR.architecture_summary(m)
display(arch)
arch.to_csv(C.OUTPUT_DIR / 'original_architecture.csv', index=False)
print('toplam parametre:', sum(p.numel() for p in m.parameters()), '| eklenen:', sum(p.numel() for p in m.new_parameters()))
del m

## Tasarım günlüğü koşuları

In [ ]:
import pandas as pd
print('hiperparametreler:', OR.convnext_hparams())
for v in OR.DESIGN_LOG:
    OR.train_variant(v, seed=42)


## Sürümlerin karşılaştırması (validation)

In [ ]:
res = pd.read_csv(C.OUTPUT_DIR / 'results.csv')
cols = [c for c in ['model_name','seed','val_macro_f1','val_balanced_accuracy','val_qwk','val_mae_grade','val_accuracy',
                     'val_distant_error_rate','param_count','epochs'] if c in res.columns]
log = res[res.model_name.isin(OR.DESIGN_LOG + ['ConvNeXtTiny__opt'])][cols]
display(log.round(4))
log.to_csv(C.OUTPUT_DIR / 'original_design_log.csv', index=False)

`outputs/results.csv`, `outputs/original_design_log.csv` ve `outputs/original_architecture.csv` dosyalarını paylaşın.